# Lesson 7: NER Evaluation & Production Deployment

## 🎯 Learning Objectives

By the end of this lesson, you will:
1. Master NER evaluation metrics (precision, recall, F1)
2. Perform comprehensive error analysis
3. Build production-ready NER systems
4. Implement monitoring and logging
5. Handle edge cases and failure modes

---

## 📚 Table of Contents

1. [Understanding NER Metrics](#1-understanding-ner-metrics)
2. [Using seqeval for Evaluation](#2-using-seqeval-for-evaluation)
3. [Error Analysis Techniques](#3-error-analysis-techniques)
4. [Production Architecture](#4-production-architecture)
5. [REST API Implementation](#5-rest-api-implementation)
6. [Monitoring & Logging](#6-monitoring--logging)
7. [Handling Edge Cases](#7-handling-edge-cases)
8. [Course Summary & Next Steps](#8-course-summary--next-steps)

---

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q seqeval transformers datasets gliner spacy matplotlib pandas
!python -m spacy download en_core_web_sm -q

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)
from seqeval.scheme import IOB2
import json
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

---

## 1. Understanding NER Metrics

### Why Standard Accuracy Doesn't Work

In NER, most tokens are "O" (outside any entity). Simple accuracy would be misleading:

```
Text:   "Barack Obama visited Paris"
Labels: [B-PER, I-PER, O, B-LOC]  (75% are entities!)

But in typical text:
"The president of the United States visited the capital city of France"
Labels: [O, O, O, O, B-LOC, I-LOC, O, O, O, O, O, B-LOC]
Only 25% are entities! Predicting all O gives 75% accuracy!
```

### Entity-Level vs Token-Level Metrics

| Metric Type | What it measures | When correct |
|-------------|-----------------|---------------|
| **Token-level** | Each token individually | Token tag matches |
| **Entity-level** | Complete entities | All tokens AND type correct |

**Entity-level is the standard** for NER evaluation.

### Precision, Recall, F1

$$\text{Precision} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}$$

$$\text{Recall} = \frac{\text{Correct Predictions}}{\text{Total Ground Truth}}$$

$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [ ]:
# Illustrate the metrics with examples

# Example predictions
y_true = [
    ['O', 'B-PER', 'I-PER', 'O', 'B-LOC', 'O', 'B-ORG', 'I-ORG'],
    ['B-PER', 'O', 'B-LOC', 'I-LOC', 'O', 'O']
]

y_pred = [
    ['O', 'B-PER', 'I-PER', 'O', 'B-LOC', 'O', 'B-ORG', 'O'],  # Missed I-ORG
    ['B-PER', 'O', 'B-LOC', 'O', 'O', 'O']  # Missed I-LOC
]

print("📊 Understanding NER Metrics\n")
print("Ground Truth vs Predictions:\n")

for i, (true, pred) in enumerate(zip(y_true, y_pred)):
    print(f"Sentence {i+1}:")
    print(f"  True: {true}")
    print(f"  Pred: {pred}")
    
    # Mark differences
    diff = ['✓' if t == p else '✗' for t, p in zip(true, pred)]
    print(f"  Match: {diff}")
    print()

In [ ]:
# Calculate metrics
print("📈 Metric Calculations:\n")

# Entity-level metrics (seqeval)
print("Entity-Level Metrics (Standard):")
print(f"  Precision: {precision_score(y_true, y_pred):.4f}")
print(f"  Recall: {recall_score(y_true, y_pred):.4f}")
print(f"  F1 Score: {f1_score(y_true, y_pred):.4f}")

# Note: An entity is only correct if ALL its tokens are correct
print("\n💡 Note: 'B-ORG I-ORG' predicted as 'B-ORG O' is WRONG at entity level")
print("   The ORG entity is incomplete, so it counts as both FP and FN")

---

## 2. Using seqeval for Evaluation

### Full Classification Report

In [ ]:
# More comprehensive example
y_true = [
    ['O', 'B-PER', 'I-PER', 'O', 'O', 'B-LOC', 'O'],
    ['B-ORG', 'I-ORG', 'O', 'B-PER', 'O', 'O', 'B-LOC'],
    ['O', 'B-PER', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'O'],
    ['B-LOC', 'O', 'B-PER', 'I-PER', 'I-PER', 'O', 'O'],
    ['O', 'O', 'B-ORG', 'O', 'B-LOC', 'I-LOC', 'O'],
]

y_pred = [
    ['O', 'B-PER', 'I-PER', 'O', 'O', 'B-LOC', 'O'],  # Perfect
    ['B-ORG', 'O', 'O', 'B-PER', 'O', 'O', 'B-LOC'],  # Missed I-ORG
    ['O', 'B-PER', 'O', 'B-ORG', 'I-ORG', 'O', 'O'],  # Missed I-ORG
    ['B-LOC', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O'],  # Missed I-PER
    ['O', 'B-LOC', 'B-ORG', 'O', 'B-LOC', 'I-LOC', 'O'],  # Extra B-LOC
]

print("📋 Full Classification Report\n")
print(classification_report(y_true, y_pred, digits=4))

In [ ]:
# Visualize per-entity performance
def plot_entity_performance(y_true, y_pred):
    """Plot precision, recall, F1 for each entity type."""
    
    # Get unique entity types
    entity_types = set()
    for seq in y_true:
        for tag in seq:
            if tag != 'O':
                entity_types.add(tag.split('-')[1])
    
    # Calculate metrics per entity
    metrics = {}
    for entity in entity_types:
        y_true_entity = [[t if t.endswith(entity) else 'O' for t in seq] for seq in y_true]
        y_pred_entity = [[t if t.endswith(entity) else 'O' for t in seq] for seq in y_pred]
        
        metrics[entity] = {
            'precision': precision_score(y_true_entity, y_pred_entity),
            'recall': recall_score(y_true_entity, y_pred_entity),
            'f1': f1_score(y_true_entity, y_pred_entity)
        }
    
    # Plot
    entities = list(metrics.keys())
    x = np.arange(len(entities))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    precision = [metrics[e]['precision'] for e in entities]
    recall = [metrics[e]['recall'] for e in entities]
    f1 = [metrics[e]['f1'] for e in entities]
    
    bars1 = ax.bar(x - width, precision, width, label='Precision', color='#2ecc71')
    bars2 = ax.bar(x, recall, width, label='Recall', color='#3498db')
    bars3 = ax.bar(x + width, f1, width, label='F1', color='#e74c3c')
    
    ax.set_ylabel('Score')
    ax.set_title('Per-Entity Type Performance')
    ax.set_xticks(x)
    ax.set_xticklabels(entities)
    ax.legend()
    ax.set_ylim(0, 1.1)
    
    # Add value labels
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.2f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.show()

plot_entity_performance(y_true, y_pred)

### Strict vs Lenient Evaluation

In [ ]:
# Different evaluation modes
from seqeval.metrics import f1_score
from seqeval.scheme import IOB2, IOBES, IOB1

# Example with boundary errors
y_true_boundary = [
    ['O', 'B-PER', 'I-PER', 'I-PER', 'O'],  # "John Paul Jones"
]

y_pred_boundary = [
    ['O', 'B-PER', 'I-PER', 'O', 'O'],  # "John Paul" (missed "Jones")
]

print("🎯 Evaluation Modes\n")
print("Ground Truth: ['O', 'B-PER', 'I-PER', 'I-PER', 'O']  → 'John Paul Jones'")
print("Prediction:   ['O', 'B-PER', 'I-PER', 'O', 'O']      → 'John Paul'")
print()

# Strict evaluation (default)
strict_f1 = f1_score(y_true_boundary, y_pred_boundary, mode='strict', scheme=IOB2)
print(f"Strict F1: {strict_f1:.4f}")
print("   → Entity boundaries must match exactly")

# Note: seqeval's default mode treats partial matches as errors
print("\n💡 In strict mode, 'John Paul' is wrong because 'John Paul Jones' is the true entity")

---

## 3. Error Analysis Techniques

### Categorizing Errors

In [ ]:
def analyze_ner_errors(y_true, y_pred, tokens_list):
    """
    Comprehensive error analysis for NER.
    
    Returns:
        Dict with error categories and examples
    """
    errors = {
        'false_negatives': [],  # Missed entities
        'false_positives': [],  # Spurious entities
        'type_errors': [],      # Wrong entity type
        'boundary_errors': [],  # Partial matches
    }
    
    for sent_idx, (true_seq, pred_seq, tokens) in enumerate(zip(y_true, y_pred, tokens_list)):
        # Extract entities from sequences
        true_entities = extract_entities(true_seq, tokens)
        pred_entities = extract_entities(pred_seq, tokens)
        
        # Find matches and errors
        true_set = set(true_entities)
        pred_set = set(pred_entities)
        
        # Perfect matches
        correct = true_set & pred_set
        
        # False negatives (missed)
        for ent in true_set - pred_set:
            errors['false_negatives'].append({
                'sentence_idx': sent_idx,
                'entity': ent,
                'context': ' '.join(tokens)
            })
        
        # False positives (spurious)
        for ent in pred_set - true_set:
            # Check if it's a type error or boundary error
            text = ent[0]
            true_texts = [e[0] for e in true_entities]
            
            if text in true_texts:
                # Same text, different type = type error
                true_type = [e[1] for e in true_entities if e[0] == text][0]
                errors['type_errors'].append({
                    'sentence_idx': sent_idx,
                    'text': text,
                    'predicted_type': ent[1],
                    'true_type': true_type,
                    'context': ' '.join(tokens)
                })
            elif any(text in t or t in text for t in true_texts):
                # Partial overlap = boundary error
                errors['boundary_errors'].append({
                    'sentence_idx': sent_idx,
                    'predicted': ent,
                    'context': ' '.join(tokens)
                })
            else:
                # No overlap = pure false positive
                errors['false_positives'].append({
                    'sentence_idx': sent_idx,
                    'entity': ent,
                    'context': ' '.join(tokens)
                })
    
    return errors

def extract_entities(tags, tokens):
    """Extract (text, type) tuples from BIO tags."""
    entities = []
    current_entity = []
    current_type = None
    
    for token, tag in zip(tokens, tags):
        if tag.startswith('B-'):
            if current_entity:
                entities.append((' '.join(current_entity), current_type))
            current_entity = [token]
            current_type = tag[2:]
        elif tag.startswith('I-') and current_entity:
            current_entity.append(token)
        else:
            if current_entity:
                entities.append((' '.join(current_entity), current_type))
                current_entity = []
                current_type = None
    
    if current_entity:
        entities.append((' '.join(current_entity), current_type))
    
    return entities

# Test data
tokens_list = [
    ['Barack', 'Obama', 'visited', 'Paris', 'yesterday'],
    ['Apple', 'Inc', 'CEO', 'Tim', 'Cook', 'spoke'],
    ['The', 'European', 'Union', 'met', 'in', 'Brussels'],
]

y_true_analysis = [
    ['B-PER', 'I-PER', 'O', 'B-LOC', 'O'],
    ['B-ORG', 'I-ORG', 'O', 'B-PER', 'I-PER', 'O'],
    ['O', 'B-ORG', 'I-ORG', 'O', 'O', 'B-LOC'],
]

y_pred_analysis = [
    ['B-PER', 'I-PER', 'O', 'B-LOC', 'O'],           # Correct
    ['B-ORG', 'O', 'O', 'B-PER', 'O', 'O'],          # Boundary errors
    ['O', 'B-LOC', 'I-LOC', 'O', 'O', 'B-LOC'],      # Type error
]

errors = analyze_ner_errors(y_true_analysis, y_pred_analysis, tokens_list)

print("🔍 Error Analysis Results\n")
print("=" * 60)

for error_type, examples in errors.items():
    print(f"\n{error_type.upper()} ({len(examples)} errors):")
    for ex in examples[:3]:  # Show first 3
        print(f"   • {ex}")

In [ ]:
# Confusion matrix for entity types
def entity_confusion_matrix(y_true, y_pred, tokens_list):
    """Create confusion matrix for entity type predictions."""
    
    confusion = defaultdict(lambda: defaultdict(int))
    
    for true_seq, pred_seq, tokens in zip(y_true, y_pred, tokens_list):
        true_ents = extract_entities(true_seq, tokens)
        pred_ents = extract_entities(pred_seq, tokens)
        
        # Match by text
        for true_text, true_type in true_ents:
            matched = False
            for pred_text, pred_type in pred_ents:
                if true_text == pred_text:
                    confusion[true_type][pred_type] += 1
                    matched = True
                    break
            if not matched:
                confusion[true_type]['MISSED'] += 1
        
        # Check for spurious predictions
        for pred_text, pred_type in pred_ents:
            if not any(pred_text == t for t, _ in true_ents):
                confusion['NONE'][pred_type] += 1
    
    return dict(confusion)

confusion = entity_confusion_matrix(y_true_analysis, y_pred_analysis, tokens_list)

print("📊 Entity Type Confusion Matrix\n")
print("(Row: True type, Column: Predicted type)\n")

# Get all types
all_types = set()
for true_type, preds in confusion.items():
    all_types.add(true_type)
    all_types.update(preds.keys())
all_types = sorted(all_types)

# Print matrix
print(f"{'True/Pred':<10}", end='')
for t in all_types:
    print(f"{t:<10}", end='')
print()

for true_type in sorted(confusion.keys()):
    print(f"{true_type:<10}", end='')
    for pred_type in all_types:
        count = confusion.get(true_type, {}).get(pred_type, 0)
        print(f"{count:<10}", end='')
    print()

---

## 4. Production Architecture

### System Design

In [ ]:
# Production-ready NER service
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from enum import Enum
import hashlib

class ModelType(Enum):
    SPACY = "spacy"
    BERT = "bert"
    GLINER = "gliner"

@dataclass
class Entity:
    text: str
    label: str
    start: int
    end: int
    confidence: float
    
    def to_dict(self) -> Dict:
        return {
            'text': self.text,
            'label': self.label,
            'start': self.start,
            'end': self.end,
            'confidence': self.confidence
        }

@dataclass
class NERResponse:
    text: str
    entities: List[Entity]
    model_used: str
    processing_time_ms: float
    request_id: str
    
    def to_dict(self) -> Dict:
        return {
            'text': self.text,
            'entities': [e.to_dict() for e in self.entities],
            'model_used': self.model_used,
            'processing_time_ms': self.processing_time_ms,
            'request_id': self.request_id
        }

class ProductionNERService:
    """
    Production-ready NER service with:
    - Multiple model backends
    - Caching
    - Logging
    - Error handling
    - Request tracking
    """
    
    def __init__(self, cache_enabled=True, log_enabled=True):
        self.models = {}
        self.cache = {} if cache_enabled else None
        self.log_enabled = log_enabled
        self.request_count = 0
        self.error_count = 0
        self.total_processing_time = 0
        
        # Initialize models lazily
        self._models_loaded = False
    
    def _load_models(self):
        """Lazy load models on first use."""
        if self._models_loaded:
            return
        
        import spacy
        from transformers import pipeline
        from gliner import GLiNER
        
        self.models['spacy'] = spacy.load('en_core_web_sm')
        self.models['bert'] = pipeline('ner', model='dslim/bert-base-NER', aggregation_strategy='simple')
        self.models['gliner'] = GLiNER.from_pretrained('urchade/gliner_medium-v2.1')
        
        self._models_loaded = True
        self._log("Models loaded successfully")
    
    def _log(self, message: str, level: str = "INFO"):
        """Simple logging."""
        if self.log_enabled:
            timestamp = datetime.now().isoformat()
            print(f"[{timestamp}] [{level}] {message}")
    
    def _generate_request_id(self, text: str) -> str:
        """Generate unique request ID."""
        timestamp = str(time.time())
        return hashlib.md5((text + timestamp).encode()).hexdigest()[:12]
    
    def _get_cache_key(self, text: str, model: str, labels: Optional[List[str]]) -> str:
        """Generate cache key."""
        labels_str = ','.join(sorted(labels)) if labels else ''
        return hashlib.md5(f"{text}:{model}:{labels_str}".encode()).hexdigest()
    
    def extract(
        self,
        text: str,
        model: str = 'bert',
        labels: Optional[List[str]] = None,
        threshold: float = 0.5
    ) -> NERResponse:
        """
        Extract entities from text.
        
        Args:
            text: Input text
            model: 'spacy', 'bert', or 'gliner'
            labels: Entity labels (required for gliner)
            threshold: Confidence threshold
        
        Returns:
            NERResponse object
        """
        self._load_models()
        
        request_id = self._generate_request_id(text)
        self.request_count += 1
        
        # Check cache
        if self.cache is not None:
            cache_key = self._get_cache_key(text, model, labels)
            if cache_key in self.cache:
                self._log(f"Cache hit for request {request_id}")
                return self.cache[cache_key]
        
        start_time = time.time()
        
        try:
            if model == 'spacy':
                entities = self._extract_spacy(text)
            elif model == 'bert':
                entities = self._extract_bert(text, threshold)
            elif model == 'gliner':
                if not labels:
                    raise ValueError("Labels required for GLiNER model")
                entities = self._extract_gliner(text, labels, threshold)
            else:
                raise ValueError(f"Unknown model: {model}")
            
            processing_time = (time.time() - start_time) * 1000
            self.total_processing_time += processing_time
            
            response = NERResponse(
                text=text,
                entities=entities,
                model_used=model,
                processing_time_ms=processing_time,
                request_id=request_id
            )
            
            # Cache result
            if self.cache is not None:
                self.cache[cache_key] = response
            
            self._log(f"Request {request_id} completed in {processing_time:.2f}ms")
            return response
            
        except Exception as e:
            self.error_count += 1
            self._log(f"Error in request {request_id}: {str(e)}", level="ERROR")
            raise
    
    def _extract_spacy(self, text: str) -> List[Entity]:
        doc = self.models['spacy'](text)
        return [
            Entity(
                text=ent.text,
                label=ent.label_,
                start=ent.start_char,
                end=ent.end_char,
                confidence=0.95  # spaCy doesn't provide confidence
            )
            for ent in doc.ents
        ]
    
    def _extract_bert(self, text: str, threshold: float) -> List[Entity]:
        results = self.models['bert'](text)
        return [
            Entity(
                text=r['word'],
                label=r['entity_group'],
                start=r['start'],
                end=r['end'],
                confidence=r['score']
            )
            for r in results
            if r['score'] >= threshold
        ]
    
    def _extract_gliner(self, text: str, labels: List[str], threshold: float) -> List[Entity]:
        results = self.models['gliner'].predict_entities(text, labels, threshold=threshold)
        return [
            Entity(
                text=r['text'],
                label=r['label'],
                start=r['start'],
                end=r['end'],
                confidence=r['score']
            )
            for r in results
        ]
    
    def get_stats(self) -> Dict:
        """Get service statistics."""
        return {
            'total_requests': self.request_count,
            'error_count': self.error_count,
            'error_rate': self.error_count / max(self.request_count, 1),
            'avg_processing_time_ms': self.total_processing_time / max(self.request_count, 1),
            'cache_size': len(self.cache) if self.cache else 0
        }

print("✅ ProductionNERService class defined")

In [ ]:
# Test the production service
service = ProductionNERService()

# Test different models
test_text = "Apple CEO Tim Cook announced new products at their Cupertino headquarters."

print("\n🧪 Testing Production NER Service\n")
print(f"Text: {test_text}\n")
print("=" * 70)

# BERT extraction
print("\n📌 BERT Model:")
response_bert = service.extract(test_text, model='bert')
for ent in response_bert.entities:
    print(f"   {ent.text:<20} → {ent.label} ({ent.confidence:.3f})")

# GLiNER extraction
print("\n📌 GLiNER Model:")
response_gliner = service.extract(
    test_text, 
    model='gliner', 
    labels=['person', 'company', 'product', 'location']
)
for ent in response_gliner.entities:
    print(f"   {ent.text:<20} → {ent.label} ({ent.confidence:.3f})")

# Test caching
print("\n📌 Testing Cache (second request):")
response_cached = service.extract(test_text, model='bert')  # Should hit cache

# Stats
print("\n📊 Service Statistics:")
stats = service.get_stats()
for key, value in stats.items():
    print(f"   {key}: {value}")

---

## 5. REST API Implementation

Here's how you would wrap this in a FastAPI service:

In [ ]:
# FastAPI implementation example (save as app.py)

fastapi_code = '''
# app.py - FastAPI NER Service

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(
    title="NER API",
    description="Named Entity Recognition Service",
    version="1.0.0"
)

# Initialize NER service (from previous code)
ner_service = ProductionNERService()

class NERRequest(BaseModel):
    text: str
    model: str = "bert"  # "bert", "spacy", "gliner"
    labels: Optional[List[str]] = None
    threshold: float = 0.5

class EntityResponse(BaseModel):
    text: str
    label: str
    start: int
    end: int
    confidence: float

class NERResponseModel(BaseModel):
    entities: List[EntityResponse]
    model_used: str
    processing_time_ms: float
    request_id: str

@app.post("/extract", response_model=NERResponseModel)
async def extract_entities(request: NERRequest):
    """
    Extract named entities from text.
    """
    try:
        response = ner_service.extract(
            text=request.text,
            model=request.model,
            labels=request.labels,
            threshold=request.threshold
        )
        return response.to_dict()
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail="Internal server error")

@app.get("/health")
async def health_check():
    """Health check endpoint."""
    return {"status": "healthy", "stats": ner_service.get_stats()}

@app.get("/models")
async def list_models():
    """List available models."""
    return {
        "models": [
            {"name": "bert", "description": "BERT-based NER (PER, ORG, LOC, MISC)"},
            {"name": "spacy", "description": "spaCy NER (18 entity types)"},
            {"name": "gliner", "description": "Zero-shot NER (custom labels)"}
        ]
    }

# Run with: uvicorn app:app --host 0.0.0.0 --port 8000
'''

print("📝 FastAPI Implementation:")
print(fastapi_code)

In [ ]:
# Example API usage (curl commands)
print("\n🔗 API Usage Examples:\n")

examples = [
    {
        "name": "Extract with BERT",
        "curl": '''curl -X POST "http://localhost:8000/extract" \
  -H "Content-Type: application/json" \
  -d '{"text": "Apple CEO Tim Cook visited Paris", "model": "bert"}' '''
    },
    {
        "name": "Extract with GLiNER (custom labels)",
        "curl": '''curl -X POST "http://localhost:8000/extract" \
  -H "Content-Type: application/json" \
  -d '{"text": "Dr. Smith prescribed Aspirin 500mg", "model": "gliner", 
       "labels": ["doctor", "medication", "dosage"]}' '''
    },
    {
        "name": "Health check",
        "curl": 'curl "http://localhost:8000/health"'
    }
]

for ex in examples:
    print(f"📌 {ex['name']}:")
    print(f"   {ex['curl']}")
    print()

---

## 6. Monitoring & Logging

In [ ]:
# Monitoring dashboard simulation
import random

class NERMonitor:
    """
    Monitor NER service performance and quality.
    """
    
    def __init__(self):
        self.metrics_history = []
        self.alerts = []
    
    def record_request(self, response: NERResponse):
        """Record metrics for a request."""
        self.metrics_history.append({
            'timestamp': datetime.now(),
            'request_id': response.request_id,
            'model': response.model_used,
            'processing_time_ms': response.processing_time_ms,
            'entity_count': len(response.entities),
            'text_length': len(response.text)
        })
        
        # Check for alerts
        if response.processing_time_ms > 1000:
            self.alerts.append({
                'type': 'SLOW_REQUEST',
                'request_id': response.request_id,
                'value': response.processing_time_ms
            })
    
    def get_summary(self) -> Dict:
        """Get monitoring summary."""
        if not self.metrics_history:
            return {}
        
        times = [m['processing_time_ms'] for m in self.metrics_history]
        entities = [m['entity_count'] for m in self.metrics_history]
        
        return {
            'total_requests': len(self.metrics_history),
            'avg_latency_ms': np.mean(times),
            'p95_latency_ms': np.percentile(times, 95),
            'p99_latency_ms': np.percentile(times, 99),
            'avg_entities_per_request': np.mean(entities),
            'alerts_count': len(self.alerts)
        }
    
    def plot_latency_distribution(self):
        """Plot latency distribution."""
        if not self.metrics_history:
            print("No data to plot")
            return
        
        times = [m['processing_time_ms'] for m in self.metrics_history]
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Histogram
        axes[0].hist(times, bins=20, edgecolor='black', alpha=0.7)
        axes[0].axvline(np.mean(times), color='red', linestyle='--', label=f'Mean: {np.mean(times):.1f}ms')
        axes[0].axvline(np.percentile(times, 95), color='orange', linestyle='--', label=f'P95: {np.percentile(times, 95):.1f}ms')
        axes[0].set_xlabel('Latency (ms)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Latency Distribution')
        axes[0].legend()
        
        # Time series
        axes[1].plot(times, marker='o', markersize=3, alpha=0.5)
        axes[1].axhline(np.mean(times), color='red', linestyle='--', alpha=0.5)
        axes[1].set_xlabel('Request #')
        axes[1].set_ylabel('Latency (ms)')
        axes[1].set_title('Latency Over Time')
        
        plt.tight_layout()
        plt.show()

# Simulate monitoring data
monitor = NERMonitor()

# Generate synthetic data
for i in range(100):
    fake_response = NERResponse(
        text="Sample text " * random.randint(5, 50),
        entities=[Entity("fake", "PERSON", 0, 4, 0.9)] * random.randint(0, 5),
        model_used=random.choice(['bert', 'gliner', 'spacy']),
        processing_time_ms=random.gauss(100, 30) + random.random() * 50,
        request_id=f"req_{i}"
    )
    monitor.record_request(fake_response)

print("📊 Monitoring Summary:\n")
summary = monitor.get_summary()
for key, value in summary.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.2f}")
    else:
        print(f"   {key}: {value}")

monitor.plot_latency_distribution()

---

## 7. Handling Edge Cases

In [ ]:
# Edge cases and how to handle them

class RobustNERExtractor:
    """
    NER extractor with robust edge case handling.
    """
    
    def __init__(self, service: ProductionNERService):
        self.service = service
        self.max_text_length = 10000
        self.chunk_size = 512
        self.overlap = 50
    
    def extract(self, text: str, **kwargs) -> NERResponse:
        """
        Extract entities with edge case handling.
        """
        # Edge case 1: Empty or whitespace text
        if not text or not text.strip():
            return NERResponse(
                text=text or "",
                entities=[],
                model_used="none",
                processing_time_ms=0,
                request_id="empty"
            )
        
        # Edge case 2: Very long text
        if len(text) > self.max_text_length:
            return self._extract_with_chunking(text, **kwargs)
        
        # Edge case 3: Text with special characters
        text = self._sanitize_text(text)
        
        # Normal extraction
        return self.service.extract(text, **kwargs)
    
    def _sanitize_text(self, text: str) -> str:
        """Clean text of problematic characters."""
        # Replace null bytes
        text = text.replace('\x00', '')
        # Normalize unicode
        import unicodedata
        text = unicodedata.normalize('NFKC', text)
        return text
    
    def _extract_with_chunking(self, text: str, **kwargs) -> NERResponse:
        """
        Process long text in overlapping chunks.
        """
        all_entities = []
        total_time = 0
        
        # Split into chunks
        chunks = self._create_chunks(text)
        
        for chunk_start, chunk_end, chunk_text in chunks:
            response = self.service.extract(chunk_text, **kwargs)
            
            # Adjust entity positions to original text
            for entity in response.entities:
                adjusted_entity = Entity(
                    text=entity.text,
                    label=entity.label,
                    start=entity.start + chunk_start,
                    end=entity.end + chunk_start,
                    confidence=entity.confidence
                )
                all_entities.append(adjusted_entity)
            
            total_time += response.processing_time_ms
        
        # Deduplicate entities from overlapping regions
        unique_entities = self._deduplicate_entities(all_entities)
        
        return NERResponse(
            text=text,
            entities=unique_entities,
            model_used=kwargs.get('model', 'bert') + '_chunked',
            processing_time_ms=total_time,
            request_id=self.service._generate_request_id(text)
        )
    
    def _create_chunks(self, text: str):
        """Create overlapping chunks."""
        chunks = []
        start = 0
        
        while start < len(text):
            end = min(start + self.chunk_size, len(text))
            
            # Try to break at sentence boundary
            if end < len(text):
                for sep in ['. ', '\n', ' ']:
                    last_sep = text[start:end].rfind(sep)
                    if last_sep > self.chunk_size // 2:
                        end = start + last_sep + 1
                        break
            
            chunks.append((start, end, text[start:end]))
            start = end - self.overlap
        
        return chunks
    
    def _deduplicate_entities(self, entities: List[Entity]) -> List[Entity]:
        """Remove duplicate entities from overlapping chunks."""
        seen = {}
        
        for entity in entities:
            key = (entity.start, entity.end, entity.label)
            if key not in seen or entity.confidence > seen[key].confidence:
                seen[key] = entity
        
        return list(seen.values())

print("✅ RobustNERExtractor class defined")

In [ ]:
# Test edge cases
robust_extractor = RobustNERExtractor(service)

edge_cases = [
    ("Empty string", ""),
    ("Whitespace only", "   \n\t  "),
    ("Unicode", "Café owner François visited München"),
    ("Special chars", "Apple's CEO—Tim Cook—spoke today."),
    ("Long text", "Google " * 1000 + "is a company."),
]

print("🧪 Edge Case Testing\n")
print("=" * 60)

for name, text in edge_cases:
    print(f"\n📌 {name}:")
    print(f"   Input length: {len(text)}")
    
    try:
        response = robust_extractor.extract(text, model='bert')
        print(f"   Entities found: {len(response.entities)}")
        print(f"   Processing time: {response.processing_time_ms:.2f}ms")
        print(f"   Status: ✅ Success")
    except Exception as e:
        print(f"   Status: ❌ Error - {str(e)}")

---

## 8. Course Summary & Next Steps

### What We Covered

| Lesson | Topic | Key Takeaways |
|--------|-------|---------------|
| 1 | NER Fundamentals | BIO tagging, entity types, evaluation basics |
| 2 | spaCy NER | Pre-trained models, entity rulers, training |
| 3 | BERT NER | Token classification, subword handling |
| 4 | GLiNER | Zero-shot NER, custom entity types |
| 5 | Fine-tuning | Dataset prep, training, hyperparameters |
| 6 | Advanced NER | NuNER, model comparison, unified pipelines |
| 7 | Production | Evaluation, API, monitoring, edge cases |

### Quick Reference: Choosing a Model

```
Need standard entities (PER, ORG, LOC)?
├── Yes → Use spaCy or pre-trained BERT NER
└── No (custom entities)
    ├── Have training data?
    │   ├── Yes → Fine-tune BERT
    │   └── No → Use GLiNER or NuNER
    └── Need long entities?
        ├── Yes → Use NuNER
        └── No → Use GLiNER (faster)
```

### Further Learning Resources

#### 📚 Research Papers
1. [BERT (Devlin et al., 2019)](https://arxiv.org/abs/1810.04805)
2. [GLiNER (Zaratiana et al., 2023)](https://arxiv.org/abs/2311.08526)
3. [NuNER (2024)](https://arxiv.org/abs/2402.15343)

#### 🔗 Documentation
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/)
- [spaCy](https://spacy.io/)
- [seqeval](https://github.com/chakki-works/seqeval)
- [GLiNER](https://github.com/urchade/GLiNER)

#### 🎓 Courses
- [Hugging Face NLP Course](https://huggingface.co/learn/nlp-course)
- [Stanford CS224N](http://web.stanford.edu/class/cs224n/)

In [ ]:
print("🎉 CONGRATULATIONS!")
print("=" * 60)
print("\nYou have completed the NER End-to-End Course!")
print("\n📝 You've learned:")
print("   ✅ NER fundamentals and tagging schemes")
print("   ✅ Using spaCy for traditional NER")
print("   ✅ BERT-based token classification")
print("   ✅ Zero-shot NER with GLiNER")
print("   ✅ Fine-tuning custom NER models")
print("   ✅ Advanced techniques with NuNER")
print("   ✅ Production deployment and monitoring")
print("\n🚀 Next steps:")
print("   • Build your own NER application")
print("   • Fine-tune models for your domain")
print("   • Contribute to open-source NER projects")
print("   • Stay updated with latest research")
print("\n📫 Happy NER-ing!")